# NATICUSdroid Android Permissions Malware Dataset

Dataset cargado desde UCI con `fetch_ucirepo(id=722)`.


# Comparación: t-SNE, t-SNE cuántico, PCA clásico y qPCA

Este notebook compara técnicas clásicas y cuánticas simuladas para reducción de dimensionalidad usando el dataset Wine.

Configuraciones:
- `t-SNE clásico`: t-SNE sobre datos estandarizados.
- `t-SNE cuántico`: t-SNE usando distancias derivadas de fidelidades entre estados con amplitude encoding.
- `t-SNE + PCA clásico`: PCA clásico como reducción previa, luego t-SNE.
- `t-SNE + qPCA`: qPCA simulada vía matriz densidad/amplitude encoding, luego t-SNE.

> Nota metodológica: qPCA real requiere una forma eficiente de implementar $e^{i\rho t}$. Aquí se construye $\rho$ explícitamente para simular el flujo completo con Qiskit y mantener el experimento reproducible en máquina local.

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

# Evita errores de joblib/loky en Windows al detectar núcleos físicos.
# Debe ejecutarse antes de llamar algoritmos de sklearn que paralelizan internamente.
os.environ.setdefault('LOKY_MAX_CPU_COUNT', '1')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (
    mean_squared_error,
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
)

from scipy.linalg import expm

from qiskit import QuantumCircuit
from qiskit.circuit.library import StatePreparation, UnitaryGate, phase_estimation
from qiskit.quantum_info import DensityMatrix, partial_trace, Pauli
from qiskit.visualization import plot_bloch_vector

RANDOM_STATE = 42
np.set_printoptions(precision=5, suppress=True)
from IPython.display import display


## 1. Carga del dataset

La función `load_dataset()` deja aislada la carga de datos. Para cambiar Wine en el futuro, basta con modificar esta función y mantener la salida `(X, y, feature_names, target_name)`.

In [ ]:
def load_dataset():
    naticusdroid_android_permissions = fetch_ucirepo(id=722)
    X = naticusdroid_android_permissions.data.features.copy()
    y = naticusdroid_android_permissions.data.targets.iloc[:, 0].to_numpy()
    feature_names = list(X.columns)
    target_name = naticusdroid_android_permissions.data.targets.columns[0]
    return X, y, feature_names, target_name

X_df, y, feature_names, target_name = load_dataset()
X_raw = X_df.to_numpy(dtype=float)

print('Dataset:', X_raw.shape)
print('Target:', target_name)
print('Clases:', np.unique(y))
X_df.head()

## 2. Preprocesamiento consistente

Todos los métodos parten de los mismos datos estandarizados. Para amplitude encoding, además se aplica padding hasta la siguiente potencia de 2 y normalización L2 por muestra.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

def next_power_of_two(n):
    return 1 if n <= 1 else 2 ** math.ceil(math.log2(n))

def pad_to_dimension(X, target_dim):
    X_padded = np.zeros((X.shape[0], target_dim), dtype=float)
    X_padded[:, : X.shape[1]] = X
    return X_padded

def normalize_rows(X):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return X / norms

amplitude_dim = next_power_of_two(X_scaled.shape[1])
n_system_qubits = int(math.log2(amplitude_dim))

X_padded = pad_to_dimension(X_scaled, amplitude_dim)
X_amp = normalize_rows(X_padded)

print('Dimensión original:', X_scaled.shape[1])
print('Dimensión amplitude encoding:', amplitude_dim)
print('Qubits de sistema:', n_system_qubits)
print('Norma primera muestra:', np.linalg.norm(X_amp[0]))


## 3. Utilidades de evaluación y visualización

In [ ]:
def run_tsne(X, metric='euclidean', perplexity=None, init='random'):
    if perplexity is None:
        perplexity = min(30, max(5, (X.shape[0] - 1) // 3))
    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        metric=metric,
        init=init,
        learning_rate='auto',
        random_state=RANDOM_STATE,
        n_jobs=1,
    )
    return tsne.fit_transform(X)

def embedding_metrics(embedding, labels):
    return {
        'Silhouette': silhouette_score(embedding, labels),
        'Harabach': calinski_harabasz_score(embedding, labels),
        'Davies-Bouldin': davies_bouldin_score(embedding, labels),
    }

def plot_embedding(ax, embedding, labels, title):
    scatter = ax.scatter(
        embedding[:, 0],
        embedding[:, 1],
        c=labels,
        cmap='viridis',
        s=38,
        alpha=0.85,
        edgecolor='k',
        linewidth=0.2,
    )
    ax.set_title(title)
    ax.set_xlabel('Dimensión 1')
    ax.set_ylabel('Dimensión 2')
    ax.grid(True, alpha=0.2)
    return scatter


## 4. Circuitos cuánticos representativos

Estos circuitos no se ejecutan en hardware real. Se usan como simulación/representación del proceso de codificación y de qPCA.

In [ ]:
def amplitude_encoding_circuit(amplitudes):
    amplitudes = np.asarray(amplitudes, dtype=complex)
    qc = QuantumCircuit(n_system_qubits, name='AmplitudeEncoding')
    qc.append(StatePreparation(amplitudes), range(n_system_qubits))
    return qc

def overlap_circuit(amplitudes_a, amplitudes_b):
    qc = QuantumCircuit(n_system_qubits, name='QuantumOverlap')
    qc.append(StatePreparation(amplitudes_a), range(n_system_qubits))
    qc.append(StatePreparation(amplitudes_b).inverse(), range(n_system_qubits))
    return qc

amp_circuit = amplitude_encoding_circuit(X_amp[0])
overlap_example = overlap_circuit(X_amp[0], X_amp[1])

print('Circuito de amplitude encoding:')
print(amp_circuit.draw(output='text'))
print('Depth:', amp_circuit.depth(), '| Width:', amp_circuit.num_qubits)

print('\nCircuito de overlap/fidelidad cuántica:')
print(overlap_example.draw(output='text'))
print('Depth:', overlap_example.depth(), '| Width:', overlap_example.num_qubits)


## 5. PCA clásico y qPCA simulada

`PCA_COMPONENTS_FOR_TSNE` controla cuántas dimensiones se usan antes de t-SNE en las variantes PCA/qPCA.

In [ ]:
PCA_COMPONENTS_FOR_TSNE = min(8, X_scaled.shape[1])
N_EVAL_QUBITS = 4

# PCA clásico
pca = PCA(n_components=PCA_COMPONENTS_FOR_TSNE, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
X_pca_reconstructed = pca.inverse_transform(X_pca)
pca_variance = pca.explained_variance_ratio_.sum()
pca_mse = mean_squared_error(X_scaled, X_pca_reconstructed)

# qPCA simulada: rho = promedio de |x><x|
rho = (X_amp.T @ X_amp) / X_amp.shape[0]
rho = (rho + rho.T) / 2

eigvals, eigvecs = np.linalg.eigh(rho)
order = np.argsort(eigvals)[::-1]
eigvals = eigvals[order]
eigvecs = eigvecs[:, order]

qpca_components = min(PCA_COMPONENTS_FOR_TSNE, amplitude_dim)
Vq = eigvecs[:, :qpca_components]
X_qpca = X_amp @ Vq
X_qpca_reconstructed = X_qpca @ Vq.T

qpca_variance = eigvals[:qpca_components].sum() / eigvals.sum()
qpca_mse = mean_squared_error(X_amp, X_qpca_reconstructed)

# Circuito QPE para representar qPCA
# NOTE: QPE on the full 512-dim amplitude space would require computing expm of a
# 512x512 complex matrix — computationally prohibitive in a notebook.
# We demonstrate QPE on a small 8x8 (3-qubit) sub-block of rho instead.
_demo_dim = min(8, rho.shape[0])
rho_demo = rho[:_demo_dim, :_demo_dim]
rho_demo = (rho_demo + rho_demo.T) / 2
n_system_qubits_demo = int(np.log2(_demo_dim))
U_rho = expm(1j * 2 * np.pi * rho_demo)
U_gate = UnitaryGate(U_rho, label='exp(i2πρ)_demo')
U_circuit = QuantumCircuit(n_system_qubits_demo, name='U_rho_demo')
U_circuit.append(U_gate, range(n_system_qubits_demo))
qpe_circuit = phase_estimation(N_EVAL_QUBITS, U_circuit)

qpca_full_circuit = QuantumCircuit(N_EVAL_QUBITS + n_system_qubits_demo, name='qPCA_QPE_demo')
qpca_full_circuit.compose(qpe_circuit, inplace=True)

print('PCA clásico varianza:', pca_variance)
print('PCA clásico MSE:', pca_mse)
print('qPCA varianza:', qpca_variance)
print('qPCA MSE:', qpca_mse)
print('\nCircuito qPCA/QPE representativo:')
print(qpca_full_circuit.draw(output='text'))
print('Depth:', qpca_full_circuit.depth(), '| Width:', qpca_full_circuit.num_qubits)


## 6. t-SNE clásico y t-SNE cuántico

Para `t-SNE cuántico`, se calcula una matriz de fidelidad entre estados con amplitude encoding:

$$K_{ij}=|\langle x_i|x_j\rangle|^2$$

y se transforma en distancia:

$$D_{ij}=\sqrt{1-K_{ij}}$$

Esa distancia alimenta `TSNE(metric='precomputed')`.

In [ ]:
# t-SNE clásico
embedding_tsne = run_tsne(X_scaled, metric='euclidean', init='pca')

# t-SNE cuántico: fidelidad simulada entre estados amplitude-encoded
fidelity_kernel = np.abs(X_amp @ X_amp.T) ** 2
quantum_distance = np.sqrt(np.maximum(0.0, 1.0 - fidelity_kernel))
np.fill_diagonal(quantum_distance, 0.0)
embedding_qtsne = run_tsne(quantum_distance, metric='precomputed', init='random')

# t-SNE + PCA clásico
embedding_tsne_pca = run_tsne(X_pca, metric='euclidean', init='pca')

# t-SNE + qPCA
embedding_tsne_qpca = run_tsne(X_qpca, metric='euclidean', init='pca')

print('Embeddings generados:')
print('t-SNE clásico:', embedding_tsne.shape)
print('t-SNE cuántico:', embedding_qtsne.shape)
print('t-SNE + PCA clásico:', embedding_tsne_pca.shape)
print('t-SNE + qPCA:', embedding_tsne_qpca.shape)


## 7. Tabla comparativa

In [ ]:
results = []

def add_result(config, components, depth, width, variance, mse, embedding):
    metrics = embedding_metrics(embedding, y)
    results.append({
        'Configuración': config,
        'Componentes': components,
        'Profundidad': depth,
        'Ancho': width,
        'Varianza': variance,
        'MSE': mse,
        'Silhouette': metrics['Silhouette'],
        'Harabach': metrics['Harabach'],
        'Davies-Bouldin': metrics['Davies-Bouldin'],
    })

add_result('t-SNE clásico', 2, np.nan, np.nan, np.nan, np.nan, embedding_tsne)
add_result('t-SNE cuántico', 2, overlap_example.depth(), overlap_example.num_qubits, np.nan, np.nan, embedding_qtsne)
add_result('t-SNE + PCA clásico', PCA_COMPONENTS_FOR_TSNE, np.nan, np.nan, pca_variance, pca_mse, embedding_tsne_pca)
add_result('t-SNE + qPCA', qpca_components, qpca_full_circuit.depth(), qpca_full_circuit.num_qubits, qpca_variance, qpca_mse, embedding_tsne_qpca)

comparison_df = pd.DataFrame(results)
comparison_df_rounded = comparison_df.copy()
for col in ['Varianza', 'MSE', 'Silhouette', 'Harabach', 'Davies-Bouldin']:
    comparison_df_rounded[col] = comparison_df_rounded[col].astype(float).round(5)

comparison_df_rounded


## 8. Visualización 2D de embeddings

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10), constrained_layout=True)

plot_embedding(axes[0, 0], embedding_tsne, y, 't-SNE clásico')
plot_embedding(axes[0, 1], embedding_qtsne, y, 't-SNE cuántico (fidelidad)')
plot_embedding(axes[1, 0], embedding_tsne_pca, y, 't-SNE + PCA clásico')
scatter = plot_embedding(axes[1, 1], embedding_tsne_qpca, y, 't-SNE + qPCA')

handles, _ = scatter.legend_elements()
fig.legend(handles, [str(c) for c in np.unique(y)], title=target_name, loc='center right', bbox_to_anchor=(1.08, 0.5))
plt.show()


## 9. Exploración de Perplexity para t-SNE clásico

In [ ]:
perplexity_values = range(30, 71, 10)
all_perplexity_results = []

num_rows = math.ceil(len(perplexity_values) / 2)
fig_perplex, axes_perplex = plt.subplots(num_rows, 2, figsize=(13, 5 * num_rows), constrained_layout=True)
axes_perplex = axes_perplex.flatten() if num_rows > 1 else [axes_perplex]

print("Ejecutando t-SNE con diferentes valores de perplexity...")

for i, p in enumerate(perplexity_values):
    print(f"  - Perplexity: {p}")
    current_embedding_tsne = run_tsne(X_scaled, metric='euclidean', perplexity=p, init='random')
    metrics = embedding_metrics(current_embedding_tsne, y)

    all_perplexity_results.append({
        'Perplexity': p,
        'Silhouette': metrics['Silhouette'],
        'Harabach': metrics['Harabach'],
        'Davies-Bouldin': metrics['Davies-Bouldin'],
    })

    # Plotting
    ax = axes_perplex[i]
    scatter = plot_embedding(ax, current_embedding_tsne, y, f't-SNE clásico (Perplexity={p})')

# Hide unused subplots if any
for j in range(len(perplexity_values), len(axes_perplex)):
    fig_perplex.delaxes(axes_perplex[j])

handles, _ = scatter.legend_elements()
fig_perplex.legend(handles, [str(c) for c in np.unique(y)], title=target_name, loc='center right', bbox_to_anchor=(1.08, 0.5))
plt.show()

perplexity_df = pd.DataFrame(all_perplexity_results)
perplexity_df_rounded = perplexity_df.copy()
for col in ['Silhouette', 'Harabach', 'Davies-Bouldin']:
    perplexity_df_rounded[col] = perplexity_df_rounded[col].astype(float).round(5)

print('\nResultados de métricas para diferentes valores de Perplexity:')
display(perplexity_df_rounded)

## 9. Esfera de Bloch

Un estado de amplitude encoding usa varios qubits. Para visualizar un qubit en la esfera de Bloch, tomamos el primer estado codificado y calculamos el estado reducido de uno de sus qubits mediante traza parcial.

In [ ]:
def reduced_bloch_vector(amplitudes, qubit_to_keep=0):
    qc = amplitude_encoding_circuit(amplitudes)
    full_density = DensityMatrix.from_instruction(qc)
    traced_out = [q for q in range(n_system_qubits) if q != qubit_to_keep]
    reduced = partial_trace(full_density, traced_out)
    rho_1q = reduced.data
    bx = np.real(np.trace(rho_1q @ Pauli('X').to_matrix()))
    by = np.real(np.trace(rho_1q @ Pauli('Y').to_matrix()))
    bz = np.real(np.trace(rho_1q @ Pauli('Z').to_matrix()))
    return [bx, by, bz]

bloch_vector = reduced_bloch_vector(X_amp[0], qubit_to_keep=0)
print('Vector de Bloch:', np.round(bloch_vector, 5))
plot_bloch_vector(bloch_vector, title='Qubit 0 reducido desde amplitude encoding')


## 10. Lectura rápida de resultados

- `Silhouette` más alto suele indicar mejor separación entre clases en el embedding.
- `Harabach` (Calinski-Harabasz) más alto suele indicar clusters más compactos y separados.
- `Davies-Bouldin` más bajo suele ser mejor.
- `Varianza` y `MSE` aplican a PCA/qPCA, no a t-SNE puro porque t-SNE no tiene una reconstrucción directa del espacio original.
- La comparación cuántica aquí es simulada: útil para investigación/prototipo, no para afirmar ventaja cuántica en hardware real.